Uso simples do pydantic, com apenas ferramentas nativas

In [1]:
%pip install -q poetry pydantic


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Para não precisar de trocar o kernel para usar poetry
%pip install -q "pydantic[email]==2.6.1" fastapi httpx


Note: you may need to restart the kernel to use updated packages.


In [3]:
from enum import auto, IntFlag
from typing import Any

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    SecretStr,
    ValidationError,
)


In [ ]:
class Role(IntFlag):
    Author = auto() # Gera valores potência de 2 (começando em 1)
    Editor = auto()
    Developer = auto()
    Admin = Author | Editor | Developer


In [ ]:
class User(BaseModel):
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field(
        examples=["example@arjancodes.com"],
        description=["The email adress of the user"],
        frozen = True # Poucos sistemas reais congelariam o email do usuário
    )
    password: SecretStr = Field(
        examples=["Password123"],
        description=["The password of the user"],
    )
    role: Role = Field(
        default=None, # Teria que ter um tratamento de erro bom para isso, para evitar o Usuário só ver um None ou erro
        description=["The role of the user"]
    )


In [6]:
def validate(data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        print("User is invalid")
        for error in e.errors():
            print(error)


In [7]:
def main() -> None:
    good_data = {
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123"
    }
    bad_data = {
        "email": "<bad_data>",
        "password": "<bad_data>"
    }

    validate(good_data)
    validate(bad_data)


In [8]:
if __name__ == '__main__':
    main()


name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=None
User is invalid
{'type': 'missing', 'loc': ('name',), 'msg': 'Field required', 'input': {'email': '<bad_data>', 'password': '<bad_data>'}, 'url': 'https://errors.pydantic.dev/2.6/v/missing'}
{'type': 'value_error', 'loc': ('email',), 'msg': 'value is not a valid email address: An email address must have an @-sign.', 'input': '<bad_data>', 'ctx': {'reason': 'An email address must have an @-sign.'}}


Implementando uma validação customizada para a **senha** e **nome do usuário**

In [ ]:
import re
import enum
import hashlib # Não guardar o texto puro
from pydantic import (
    field_validator,
    model_validator,
)

# Senha minlen = 8; com números, letras maiusculas e minusculas
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")
# Nome minlen = 2; com letras maiusculas e minusculas
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")


In [ ]:
class Role(IntFlag):  # auto() faria o mesmo, mas assim fica mais explicito
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8


In [ ]:
class User(BaseModel):
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field(
        examples=["example@arjancodes.com"],
        description=["The email adress of the user"],
        frozen = True # Poucos sistemas reais congelariam o email do usuário
    )
    password: SecretStr = Field(
        examples=["Password123"],
        description=["The password of the user"],
    )
    role: Role = Field(
        default=None,
        description=["The role of the user"]
    )

    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v
    
    @field_validator("role", mode="before") # Executa antes da conversão, permitindo strings, int puro, etc
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f'Role is invalid, please use one of the following: {", ".join([x.name for x in Role])}'
            )
    
    @model_validator(mode="before")
    @classmethod
    def validate_user(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v: # Garantir presença de nome e senha
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold(): # Garantir a não presença do nome na senha
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]): # Validar senha pelo REGEX
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest() # Criptografia da senha
        return v


In [ ]:
def main() -> None:
    test_data = dict(
        good_data={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Admin",
        },
        bad_role={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Password123",
            "role": "Programmer", # Não existe (Author | Editor | Admin | SuperAdmin)
        },
        bad_data={
            "name": "Arjan",
            "email": "bad email", # email invalido (sem @ e .com)
            "password": "bad password", # Senha fraca: não tem numeros ou letras maiusculas
        },
        bad_name={
            "name": "Arjan<-_->", # Nome com caracteres invalidos (algo além de letras do alfabeto latino)
            "email": "example@arjancodes.com",
            "password": "Password123",
        },
        duplicate={
            "name": "Arjan",
            "email": "example@arjancodes.com",
            "password": "Arjan123", # Senha fraca: tem nome do usuário
        },
        missing_data={ # Sem campo 'name' (obrigatório)
            "email": "<bad data>", # email invalido (sem @ e .com)
            "password": "<bad data>", # Senha fraca: não tem numeros ou letras maiusculas
        },
    )

    for example_name, data in test_data.items():
        print(example_name)
        validate(data)


In [13]:
if __name__ == '__main__':
    main()


good_data
name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=<Role.Admin: 4>
bad_role
User is invalid
{'type': 'value_error', 'loc': ('role',), 'msg': 'Value error, Role is invalid, please use one of the following: Author, Editor, Admin, SuperAdmin', 'input': 'Programmer', 'ctx': {'error': ValueError('Role is invalid, please use one of the following: Author, Editor, Admin, SuperAdmin')}, 'url': 'https://errors.pydantic.dev/2.6/v/value_error'}
bad_data
User is invalid
{'type': 'value_error', 'loc': (), 'msg': 'Value error, Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number', 'input': {'name': 'Arjan', 'email': 'bad email', 'password': 'bad password'}, 'ctx': {'error': ValueError('Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number')}, 'url': 'https://errors.pydantic.dev/2.6/v/value_error'}
bad_name
User is invalid
{'type': 'value_error', 'loc': ('name',), 'msg': 'Value error, Name is invalid, m

In [ ]:
from typing import Self # Representar a própria classe como tipo
from pydantic import (
    field_serializer,
    model_serializer,
)


In [15]:
class Role(IntFlag):
    User = 0
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8


In [ ]:
class User(BaseModel):
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field(
        examples=["example@arjancodes.com"],
        description=["The email adress of the user"],
        frozen = True # Poucos sistemas reais congelariam o email do usuário
    )
    password: SecretStr = Field(
        examples=["Password123"],
        description=["The password of the user"],
        exclude=True, # Garantir que a senha não vai ser escrita na serialização
    )
    role: Role = Field(
        default=None,
        description=["The role of the user"]
    )

    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v
    
    @field_validator("role", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f'Role is invalid, please use one of the following: {", ".join([x.name for x in Role])}'
            )
    
    @model_validator(mode="before")
    @classmethod
    def validate_user(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v
    
    @model_validator(mode="after") # Roda depois de todas as validações, então os valores já estão normalizados
    def validate_user_post(self, v: Any) -> Self:
        if self.role == Role.Admin and self.name != "Arjan":
            raise ValueError("Only Arjan can be an admin")
        return self

    @field_serializer("role", when_used="json")
    @classmethod
    def serialize_role(cls, v) -> str:
        return v.name # Retornar o nome, não o valor int

    @model_serializer(mode="wrap", when_used="json")
    def serialize_user(self, serializer, info) -> dict[str, Any]:
        # info é um filtro simples, se nenhum estiver presente vai só nome e cargo
        if not info.include and not info.exclude:
            return {"name": self.name, "role": self.role.name}
        return serializer(self) # serialização do pydantic


In [17]:
def main() -> None:
    data = {
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
        "role": "Admin",
    }

    user = User.model_validate(data)
    if user:
        print(
            "The serializer that returns a dict:",
            user.model_dump(),
            sep="\n",
            end="\n\n",
        )
        print(
            "The serializer that returns a JSON string:",
            user.model_dump(mode="json"),
            sep="\n",
            end="\n\n",
        )
        print(
            "The serializer that returns a json string, excluding the role:",
            user.model_dump(exclude=["role"], mode="json"),
            sep="\n",
            end="\n\n",
        )
        print("The serializer that encodes all values to a dict:", dict(user), sep="\n")


In [18]:
if __name__ == '__main__':
    main()


The serializer that returns a dict:
{'name': 'Arjan', 'email': 'example@arjancodes.com', 'role': <Role.Admin: 4>}

The serializer that returns a JSON string:
{'name': 'Arjan', 'role': 'Admin'}

The serializer that returns a json string, excluding the role:
{'name': 'Arjan', 'email': 'example@arjancodes.com'}

The serializer that encodes all values to a dict:
{'name': 'Arjan', 'email': 'example@arjancodes.com', 'password': SecretStr('**********'), 'role': <Role.Admin: 4>}


Objetivo agora é usar **fastapi**

In [19]:
from datetime import datetime
from typing import Optional
from uuid import uuid4

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, EmailStr, Field, field_serializer, UUID4

app = FastAPI()


In [ ]:
# 'max_items' está em desuso, por isso foi substituido por 'max_length'
class User(BaseModel):
    model_config = {
        "extra": "forbid", # Rejeitar o que não for declaramos, melhora previsibilidade
    }
    __users__ = []
    name: str = Field(..., description="Name of the user")
    email: EmailStr = Field(..., description="Email address of the user")
    friends: list[UUID4] = Field( # Cada instancia de friends vai ser uma lista desde sua crição
        default_factory=list, max_length=500, description="List of friends"
    )
    blocked: list[UUID4] = Field( # Cada instancia de blocked vai ser uma lista desde sua crição
        default_factory=list, max_length=500, description="List of blocked users"
    )
    signup_ts: Optional[datetime] = Field(
        default_factory=datetime.now, description="Signup timestamp", kw_only=True
    )
    id: UUID4 = Field(
        default_factory=uuid4, description="Unique identifier", kw_only=True
    )

    @field_serializer("id", when_used="json")
    def serialize_id(self, id: UUID4) -> str:
        return str(id)


In [22]:
@app.get("/users", response_model=list[User])
async def get_users() -> list[User]:
    return list(User.__users__)


@app.post("/users", response_model=User)
async def create_user(user: User):
    User.__users__.append(user)
    return user


@app.get("/users/{user_id}", response_model=User)
async def get_user(user_id: UUID4) -> User | JSONResponse:
    try:
        return next((user for user in User.__users__ if user.id == user_id))
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "User not found"})
    


In [23]:
def main() -> None:
    with TestClient(app) as client:
        for i in range(5):
            response = client.post(
                "/users",
                json={"name": f"User {i}", "email": f"example{i}@arjancodes.com"},
            )
            assert response.status_code == 200
            assert (
                response.json()["name"] == f"User {i}"
            ), "The name of the user should be User {i}"
            assert response.json()["id"], "The user should have an id"

            user = User.model_validate(response.json())
            assert str(user.id) == response.json()["id"], "The id should be the same"
            assert user.signup_ts, "The signup timestamp should be set"
            assert user.friends == [], "The friends list should be empty"
            assert user.blocked == [], "The blocked list should be empty"

        response = client.get("/users")
        assert response.status_code == 200, "Response code should be 200"
        assert len(response.json()) == 5, "There should be 5 users"

        response = client.post(
            "/users", json={"name": "User 5", "email": "example5@arjancodes.com"}
        )
        assert response.status_code == 200
        assert (
            response.json()["name"] == "User 5"
        ), "The name of the user should be User 5"
        assert response.json()["id"], "The user should have an id"

        user = User.model_validate(response.json())


In [24]:
if __name__ == '__main__':
    main()
